# USDA Cropland Data Layer (CDL) — What Grew Where, Every Year

**What it is.** An annual, crop-specific satellite land-cover map of the continental U.S.
For any point you can read the **crop that was grown each year** — i.e. reconstruct a field's
**rotation history** — and for any county get **acreage by crop**.

| | |
|---|---|
| **Coverage** | Continental U.S. |
| **History** | Nationwide 2008–present (10 m from 2024; 30 m before) |
| **Cost / key** | **Free · no key** (CropScape / CroplandCROS service, hosted at GMU) |
| **Docs** | https://nassgeodata.gmu.edu/CropScape/ · https://croplandcros.scinet.usda.gov/ |

**Why it's core to Tracera.** Rotation history predicts pest/disease pressure, nitrogen
credits, and expected yield. Knowing a field ran corn→corn→soybeans is a real agronomic signal.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — crop at the field this year (point → Albers reprojection)

In [2]:
from pyproj import Transformer
import re

CDL = "https://nassgeodata.gmu.edu/axis2/services/CDLService"
# CDL rasters are in Albers Equal Area (EPSG:5070); GetCDLValue needs x/y in metres.
_to_albers = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)

def cdl_value(year, lat=LAT, lon=LON):
    x, y = _to_albers.transform(lon, lat)
    r = requests.get(f"{CDL}/GetCDLValue", params={"year": year, "x": x, "y": y}, timeout=90)
    r.raise_for_status()
    m = re.search(r'value: (\d+), category: "([^"]+)"', r.text)
    return {"year": year, "value": int(m.group(1)), "crop": m.group(2)} if m else None

print(cdl_value(2023))

{'year': 2023, 'value': 5, 'crop': 'Soybeans'}


### 2. Core query — rotation history at the field (2015 → latest)

In [3]:
history = pd.DataFrame([cdl_value(y) for y in range(2015, 2025)])
print("Crop rotation at the sample field:")
history.set_index("year")["crop"]

Crop rotation at the sample field:


year
2015    Soybeans
2016        Corn
2017        Corn
2018        Corn
2019    Soybeans
2020        Corn
2021        Corn
2022        Corn
2023    Soybeans
2024        Corn
Name: crop, dtype: str

### 3. Core query — county crop acreage (GetCDLStat → CSV histogram)

In [4]:
import io

def cdl_county_acreage(year, fips=FIPS):
    r = requests.get(f"{CDL}/GetCDLStat", params={"year": year, "fips": fips, "format": "csv"}, timeout=120)
    url = re.search(r"<returnURL>(.*?)</returnURL>", r.text).group(1)
    df = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))
    df.columns = [c.strip() for c in df.columns]
    return df.sort_values("Acreage", ascending=False)

acres = cdl_county_acreage(2023)
print("Top crops in the county by acreage, 2023:")
acres.head(8)

Top crops in the county by acreage, 2023:


,Value,Category,Count,Acreage
0,1,Corn,673189,149713.6
2,5,Soybeans,535675,119131.3
31,176,Grass/Pasture,180624,40169.8
22,121,Developed/Open Space,64222,14282.6
27,141,Deciduous Forest,53128,11815.4
23,122,Developed/Low Intensity,51462,11444.9
24,123,Developed/Medium Intensity,34649,7705.8
32,190,Woody Wetlands,15622,3474.2


### Notes & how to extend
- **Category codes:** 1 = Corn, 5 = Soybeans, 24 = Winter Wheat, 176 = Grass/Pasture, etc.
  (full legend in the CDL metadata).
- **Field polygons:** `GetCDLFile(year, fips|bbox)` returns a URL to a clipped GeoTIFF you can
  zonal-stat against a field boundary for a true per-field rotation.
- **Rate limits:** the GMU service is public and can be slow — cache results to `data_cache/`.
- **Alternative backend:** the same CDL rasters live in Google Earth Engine
  (`USDA/NASS/CDL`) if you prefer that for large areas.